# 02 · Modelado supervisado y no supervisado
Atacamos el mismo insight (sentimiento) por dos ángulos complementarios:
- **Supervisado:** clasificar el *tono* del post (3 modelos comparados).
- **No supervisado:** descubrir los *temas* de conversación (3 modelos comparados).

In [ ]:
# === Setup reproducible (Google Colab) ===
import sys, os, subprocess, glob
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

hits = glob.glob("/content/drive/MyDrive/**/src/config.py", recursive=True)
if not hits:
    raise FileNotFoundError(
        "No se encontro src/config.py en tu Drive. Verifica que subiste "
        "la carpeta completa del proyecto (con la carpeta src/ y sus .py)."
    )
root = Path(hits[0]).parent.parent
os.environ["PROJECT_ROOT"] = str(root)
sys.path.insert(0, str(root / "src"))

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "scikit-learn", "pandas", "numpy", "matplotlib", "seaborn"],
               check=False)

from config import set_global_seed, ensure_dirs, SEED
set_global_seed(SEED)   # fija todas las semillas
ensure_dirs()           # crea carpetas de salida
print(f"Entorno listo. Raiz del proyecto: {root} | SEED={SEED}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Entorno listo. Raiz del proyecto: /content/drive/MyDrive/Programación Ev2/Adolfo | SEED=42


In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from data_preprocessing import load_dataset, get_text_target
from model_training import get_supervised_models

df = load_dataset()
X, y = get_text_target(df)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y)
print('train:', X_train.shape[0], '| test:', X_test.shape[0])

train: 9600 | test: 2400


## 1. Supervisado — entrenar 3 modelos
Tres familias distintas: lineal (LogReg), probabilístico (NaiveBayes) y de árboles (RandomForest). Cada uno es un Pipeline TF-IDF + clasificador.

In [ ]:
from model_evaluation import evaluate_supervised
sup_models = get_supervised_models()
tabla_sup = evaluate_supervised(sup_models, X_train, y_train, X_test, y_test, cv=5)
tabla_sup

,cv_f1_mean,cv_f1_std,test_accuracy,test_f1_macro,train_time_s
LogReg,0.9421,0.0024,0.9342,0.9366,16.78
NaiveBayes,0.9353,0.0056,0.9346,0.9362,5.69
RandomForest,0.9385,0.0047,0.9279,0.9308,22.30


Los tres rondan F1≈0.93–0.94. La diferencia real es el **costo**: RandomForest tarda ~9× más para no mejorar. Guardamos la comparación.

In [ ]:
from config import METRICS_DIR, MODELS_DIR
import joblib
tabla_sup.to_csv(METRICS_DIR / 'supervised_comparison.csv')
best_name = tabla_sup.index[0]
best_model = sup_models[best_name]
joblib.dump(best_model, MODELS_DIR / 'best_supervised.joblib')
print('Mejor modelo supervisado:', best_name)

Mejor modelo supervisado: LogReg


## 2. No supervisado — descubrir temas
Reducimos el texto con TF-IDF + TruncatedSVD (PCA para texto disperso) y comparamos 3 algoritmos de clustering.

In [ ]:
from model_training import build_text_features, get_unsupervised_models
from model_evaluation import evaluate_clustering

# Submuestra para los metodos que no escalan (jerarquico/DBSCAN)
sub = df.sample(3000, random_state=SEED).reset_index(drop=True)
feat = build_text_features()
X_red = feat.fit_transform(sub['text_content'].fillna(''))
y_sent = sub['sentiment_label']

rows = {}
for name, model in get_unsupervised_models(n_clusters=3).items():
    labels = model.fit_predict(X_red)
    rows[name] = evaluate_clustering(labels, X_red, y_sent)
import pandas as pd
tabla_uns = pd.DataFrame(rows).T
tabla_uns

,n_clusters,silhouette,ARI_vs_sentiment,NMI_vs_sentiment
KMeans,3.0,0.0689,0.0031,0.0015
Agglomerative,3.0,0.0204,-0.0043,0.0019
DBSCAN,1.0,NaN,0.0039,0.0032


**Hallazgo clave:** el ARI vs sentimiento es ≈0. El clustering **no** agrupa por tono — agrupa por *estructura/tema* del mensaje. No contradice al supervisado: responde otra pregunta (de qué se habla, no con qué tono).

In [ ]:
# Buscar el numero natural de grupos por silhouette (sobre todo el dataset)
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
feat_full = build_text_features()
X_full = feat_full.fit_transform(df['text_content'].fillna(''))
for k in [3,4,5,6,8]:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10).fit(X_full)
    s = silhouette_score(X_full, km.labels_, sample_size=3000, random_state=SEED)
    print(f'k={k}: silhouette={s:.3f}')

k=3: silhouette=0.075
k=4: silhouette=0.084
k=5: silhouette=0.096
k=6: silhouette=0.105
k=8: silhouette=0.103


In [ ]:
tabla_uns.to_csv(METRICS_DIR / 'unsupervised_comparison.csv')
print('Comparación de clustering guardada.')

Comparación de clustering guardada.


**Conclusión 02:** mejor supervisado = LogReg (rápido e interpretable); mejor no supervisado = KMeans (escala y silhouette estable). En 03 los interpretamos en profundidad.